# Save L0 configuration files

This notebook saves L0 configuration files as assets of catalog items, so they can be used later by processing flows as if they were ADFS files.

In [ ]:
import json
from datetime import datetime, timezone
from IPython.display import Markdown
from pathlib import Path
from pystac import Asset, Collection, Extent, Item, SpatialExtent, TemporalExtent
from requests import HTTPError
from rs_common import prefect_utils
from starlette import status

OWNER = "copernicus"
COLLECTION = "configuration"
NOW = datetime.now(timezone.utc)

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
# Reload the global vars again
from resources.utils import *

In [ ]:
#catalog_client.remove_collection(COLLECTION, owner_id=OWNER) # FOR TESTING ONLY, SHOULD BE COMMENTED

print(f"Get or create collection '{OWNER}:{COLLECTION}':")
try:
    collection = Collection(
        id=COLLECTION,
        description="Configuration files",
        extent=Extent(
            SpatialExtent([[-180, -90, 180, 90]]),
            TemporalExtent([[NOW, None]]),
        ),
    )
    response = catalog_client.add_collection(collection, owner_id=OWNER)    
    print(response.status_code)
    
except HTTPError as error:
    # Ignore 409 / conflict error (the collection already existed)
    if error.response.status_code != status.HTTP_409_CONFLICT:
        raise

saved_catalog = catalog_client.get_collection(COLLECTION, owner_id=OWNER)
print(json.dumps(saved_catalog.to_dict(), indent=2))

In [ ]:
def save_adf_l0cfg(filename: str):
    """Save s1_adf_l0cfg.json or s3_adf_l0cfg.json"""

    without_ext = Path(filename).stem

    local_path = Path("./init-catalog-l0") / filename
    relative_from_home = str(local_path.resolve().relative_to(Path.home(), walk_up=True))
    display(Markdown(f"Save file as catalog item: [{relative_from_home}]({local_path})"))

    s3_path = f"s3://rs-dev-cluster-catalog/{OWNER}/{COLLECTION}/{filename}"
    print(f"Save file to bucket: {s3_path}")
    prefect_utils.s3_upload_file(local_path, s3_path, _sync=True)
    
    # Build the stac item
    infinity = datetime(9999, 12, 31, 23, 59, 59, tzinfo=timezone.utc)
    item = Item(
        id=without_ext,
        geometry=None,
        bbox=None,
        datetime=NOW,
        properties={
            "product:type": without_ext,
            "expires": infinity.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "start_datetime": NOW.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end_datetime": infinity.strftime("%Y-%m-%dT%H:%M:%SZ"),
        },
        assets={
            without_ext: Asset(
                href=s3_path, 
                media_type="application/json", 
                title=filename
            )
        },
    )
    item.validate()

    # catalog_client.remove_item(COLLECTION, without_ext, owner_id=OWNER) # FOR TESTING ONLY, SHOULD BE COMMENTED
    
    print("Get or create the stac item in the catalog collection:")
    try:
        response = catalog_client.add_item(COLLECTION, item, owner_id=OWNER)    
        print(response.status_code)
        
    except HTTPError as error:
        # Ignore 409 / conflict error (the item already existed)
        if error.response.status_code != status.HTTP_409_CONFLICT:
            raise

    saved_item = catalog_client.get_item(COLLECTION, without_ext, owner_id=OWNER)
    print(json.dumps(saved_item.to_dict(), indent=2))

save_adf_l0cfg("s1_adf_l0cfg.json")
save_adf_l0cfg("s3_adf_l0cfg.json")